# ✨ 39 · Generative design of proteins and molecules
### Design as an inverse problem; sampling new sequences from an evolutionary model; optimisation with an oracle (simulated annealing, genetic algorithms); model-based optimisation and why it drifts out of distribution; inverse folding in miniature; de novo molecule design with a surrogate; responsible design

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 39 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/39_generative_design.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, scikit-learn, PyTorch (CPU), Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 13 mixtures/EM, 21 MCMC and simulated annealing, 22 neural networks, 23 model selection); Modules 29 (Gaussian processes, Bayesian optimisation) and 37 (protein sequences / Potts models). Module 28 (diffusion) and 40 (molecular ML) are helpful and forward-referenced. |
| **Estimated time** | 195 min (reading + running + the ✏️ exercises) |
| **Builds on** | [13 · Mixture models and EM](13_mixture_models_em.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [23 · Model selection](23_model_selection.ipynb), [29 · Gaussian processes and Bayesian optimisation](29_gaussian_processes_bayesian_optimization.ipynb), [37 · Protein sequences and language models](37_protein_sequences_language_models.ipynb) |
| **Research frontier covered** | generative protein design (sampling, inverse folding, diffusion pipelines), model-based optimisation and its pitfalls, de novo molecule generation, multi-objective design, biosecurity — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** by code in this notebook, so every design can be scored against a known truth. Proteins: a
> Potts model of a protein family, sampled in-notebook with `sample_chemistry_family` on the contact map of
> `simulate_helix_bundle` — its fields $h$ and couplings $J$ follow three shared physico-chemical rules, so the true
> `potts_score` is our **oracle** and a small neural model has structure to learn (this is the structured sampler from
> Module 24; the toolkit's `simulate_protein_family` uses *random* couplings that small models cannot learn). Molecules:
> the toolkit's `simulate_molecules` library with its noise-free `oracle` for true potency and solubility. Nothing is
> downloaded, and nothing here is a real drug, a real pathogen sequence, or a laboratory protocol. Section 8 treats the
> dual-use question at a deliberately conceptual level.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Frame** design as the inverse of prediction, estimate the astronomical size of sequence and chemical space, and explain the design–build–test–learn loop.
2. **Sample** new sequences from an evolutionary (Potts) model at a temperature and **measure** the fitness–diversity trade-off against the ground-truth oracle.
3. **Optimise** sequences with an oracle using **simulated annealing** ([Module 21](21_bayesian_learning_mcmc.ipynb)) and a **genetic algorithm**, and compare their efficiency per oracle call.
4. **Demonstrate** the central failure of model-based optimisation — a learned surrogate's predictions soar while true fitness falls as designs drift out of distribution — and **fix** it with a trust region and a pessimistic (uncertainty-penalised) objective ([Module 29](29_gaussian_processes_bayesian_optimization.ipynb)).
5. **Build** inverse folding in miniature: predict a residue from its structural neighbours, measure **sequence recovery**, and decode novel sequences that the oracle scores as foldable.
6. **Design** small molecules with a random-forest surrogate and a genetic algorithm over scaffolds and substituents, **score** them with the ground-truth oracle, and **trade off** potency against solubility on a Pareto front.
7. **Describe** modern production pipelines (RFdiffusion → ProteinMPNN → AlphaFold filtering; protein language models; REINVENT-style molecule generators) and how designs are evaluated.
8. **Discuss** the dual-use and biosecurity dimensions of generative design responsibly, and the safeguards the field has adopted.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Design as an inverse problem | forward vs inverse; search-space sizes; the DBTL loop | 15 min |
| 2 · Sampling from an evolutionary model | MCMC at temperature T; 🧪 fitness vs diversity vs novelty | 25 min |
| 3 · Optimisation with an oracle | simulated annealing and a genetic algorithm; efficiency | 25 min |
| 4 · Model-based optimisation & its pitfalls | surrogate drift out of distribution; 🧪 trust region & pessimism | 35 min |
| 5 · Inverse folding in miniature | residue from neighbours; 🧪 recovery; decode & score designs | 30 min |
| 6 · Molecule design | RF surrogate + genetic algorithm; 🧪 oracle; Pareto of potency vs solubility | 30 min |
| 7 · Modern pipelines | RFdiffusion, ProteinMPNN, language-model design; evaluation | 10 min |
| 8 · Responsible design | dual use, safeguards, oversight — conceptual | 10 min |
| 9 · 🧭 From toy to real data | ProteinMPNN, RFdiffusion, REINVENT, benchmarks, synthesis | 7 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 8 min |

## 🧫 The biological question

For four billion years, evolution has been the only author of proteins and the chemistry of life has been read, not
written. Directed evolution and medicinal chemistry let us *edit* what nature provides. The question this module asks is
more radical: **can we *write* new proteins and new molecules — sequences and structures that never existed — to do a job
we specify?** Design an enzyme for a reaction evolution never needed, a binder for a chosen target, a drug with a chosen
potency and solubility.

Two things make this hard, and both are the point of the module. First, the spaces are **astronomically large**: a
100-residue protein has $20^{100}\approx10^{130}$ possible sequences; drug-like chemical space is estimated at $10^{60}$
molecules. We cannot enumerate, only search. Second, our search is guided by **imperfect models**. A learned fitness
predictor is accurate near the data it was trained on and unreliable far away — exactly where optimisation wants to go.
Pushing a model too hard produces designs it *scores* brilliantly and reality rejects. Learning to search huge spaces
*without* being fooled by our own models is the craft of generative design.

The field moved fast. Rosetta-era physics-based design built novel folds and binders (Huang, Boyken & Baker 2016).
Deep learning then transformed it: **ProteinMPNN** writes sequences for a given backbone (Dauparas *et al.* 2022),
**RFdiffusion** generates new backbones by denoising (Watson *et al.* 2023), protein **language models** generate
functional sequences (Madani *et al.* 2023; Hayes *et al.* 2025), and evolution-based models design working enzymes by
sampling (Russ *et al.* 2020). For molecules, generative models from variational autoencoders (Gómez-Bombarelli *et al.*
2018) to reinforcement-learning and genetic approaches (Jensen 2019) propose candidates that are then filtered and made.

We will build the core of each idea on **simulated** data with a known oracle, so we can see designs succeed *and* catch
our models lying to us. We close on the responsibility that comes with writing biology — the dual-use question — at a
conceptual level.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.linalg import cho_solve, cholesky, solve_triangular
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                               # PyTorch has its own RNG
torch.set_num_threads(min(2, torch.get_num_threads()))  # tiny models; be polite on a shared CPU
T_START = time.time()
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads(), "| device: cpu")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use `simulate_helix_bundle` for a structure and contact map, `potts_score` as the protein oracle, and the molecule simulator with its noise-free `oracle`, `assemble_molecule`, the descriptor and fingerprint functions and `tanimoto` for the drug-design section. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


_AR = 1.5


MOLECULE_SCAFFOLDS = {
    "benzene": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)], [0, 1, 2, 3]),
    "pyridine": ([("N", 1)] + [("C", 1)] * 5, [(i, (i + 1) % 6, _AR) for i in range(6)], [2, 3, 4]),
    "pyrimidine": ([("N", 1), ("C", 1), ("N", 1), ("C", 1), ("C", 1), ("C", 1)],
                   [(i, (i + 1) % 6, _AR) for i in range(6)], [1, 3, 4]),
    "thiophene": ([("S", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "furan": ([("O", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "cyclohexane": ([("C", 0)] * 6, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 1, 2, 3]),
    "piperidine": ([("N", 0)] + [("C", 0)] * 5, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 2, 3]),
    "morpholine": ([("O", 0), ("C", 0), ("C", 0), ("N", 0), ("C", 0), ("C", 0)],
                   [(i, (i + 1) % 6, 1) for i in range(6)], [3, 1, 2]),
    "naphthalene": ([("C", 1)] * 10, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR), (5, 0, _AR),
                                      (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR), (9, 5, _AR)], [0, 1, 2, 7]),
    "quinoline": ([("N", 1)] + [("C", 1)] * 9, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR),
                                                (5, 0, _AR), (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR),
                                                (9, 5, _AR)], [1, 2, 7]),
}


MOLECULE_SUBSTITUENTS = {      # attachment atom = index 0
    "methyl": ([("C", 0)], []),
    "ethyl": ([("C", 0), ("C", 0)], [(0, 1, 1)]),
    "hydroxyl": ([("O", 0)], []),
    "amino": ([("N", 0)], []),
    "fluoro": ([("F", 0)], []),
    "chloro": ([("Cl", 0)], []),
    "methoxy": ([("O", 0), ("C", 0)], [(0, 1, 1)]),
    "carboxyl": ([("C", 0), ("O", 0), ("O", 0)], [(0, 1, 2), (0, 2, 1)]),
    "amide": ([("C", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 1)]),
    "nitrile": ([("C", 0), ("N", 0)], [(0, 1, 3)]),
    "trifluoromethyl": ([("C", 0), ("F", 0), ("F", 0), ("F", 0)], [(0, 1, 1), (0, 2, 1), (0, 3, 1)]),
    "sulfonamide": ([("S", 0), ("O", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 2), (0, 3, 1)]),
    "dimethylamino": ([("N", 0), ("C", 0), ("C", 0)], [(0, 1, 1), (0, 2, 1)]),
    "acetyl": ([("C", 0), ("O", 0), ("C", 0)], [(0, 1, 2), (0, 2, 1)]),
    "phenyl": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)]),
}


_VALENCE = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1}


_MASS = {"C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "H": 1.008}


def molecule_to_smiles(atoms, bonds) -> str:
    """Write a molecular graph as a SMILES string (depth-first, ring closures as digits; not canonical)."""
    n = len(atoms)
    nbr = {i: [] for i in range(n)}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    for i in nbr:
        nbr[i].sort()
    parent, order, closures, seen = {0: None}, [], {}, set()

    def visit(a):                                    # first pass: tree edges vs ring-closure edges
        seen.add(a)
        order.append(a)
        for b, o in nbr[a]:
            if b not in seen:
                parent[b] = a
                visit(b)
            elif b != parent[a] and (min(a, b), max(a, b)) not in closures:
                closures[(min(a, b), max(a, b))] = o
    visit(0)
    ring_at = {i: [] for i in range(n)}
    for k, (pair, o) in enumerate(closures.items()):
        for a in pair:
            ring_at[a].append((k + 1, o, pair))

    def bond_symbol(o, a, b):
        if o == 2:
            return "="
        if o == 3:
            return "#"
        return "-" if (o == 1 and atoms[a][1] and atoms[b][1]) else ""

    def write(a):
        el, arom = atoms[a]
        s = el.lower() if arom else el
        for digit, o, pair in ring_at[a]:
            first = a == min(pair, key=order.index)
            s += (bond_symbol(o, *pair) if first else "") + (str(digit) if digit < 10 else f"%{digit}")
        kids = [(b, o) for b, o in nbr[a] if parent.get(b) == a]
        for idx, (b, o) in enumerate(kids):
            sub = bond_symbol(o, a, b) + write(b)
            s += sub if idx == len(kids) - 1 else f"({sub})"
        return s
    return write(0)


def molecule_descriptors(atoms, bonds) -> dict:
    """Simple descriptors: heavy atoms, implicit H, molecular weight, H-bond donors/acceptors,
    aromatic atoms, rings and a simplified (Crippen-like) octanol–water partition coefficient."""
    used = np.zeros(len(atoms))
    for i, j, o in bonds:
        used[i] += o
        used[j] += o
    n_h = np.array([max(0, int(np.floor(_VALENCE[el] - used[k] + 1e-9))) if el in ("C", "N", "O") else 0
                    for k, (el, _) in enumerate(atoms)])
    els = [el for el, _ in atoms]
    mw = sum(_MASS[e] for e in els) + _MASS["H"] * n_h.sum()
    hbd = sum(1 for k, e in enumerate(els) if e in ("N", "O") and n_h[k] > 0)
    hba = sum(1 for e in els if e in ("N", "O"))
    contrib = {"C": 0.25, "N": -0.75, "O": -0.55, "S": 0.4, "F": 0.35, "Cl": 0.7}
    logp = sum(contrib[e] + (0.12 if a else 0) for e, a in atoms) - 0.3 * hbd
    return dict(heavy_atoms=len(atoms), n_hydrogens=int(n_h.sum()), mol_weight=round(mw, 2), hbd=hbd, hba=hba,
                aromatic_atoms=int(sum(a for _, a in atoms)), rings=len(bonds) - len(atoms) + 1,
                logp=round(logp, 3))


def molecular_fingerprint(atoms, bonds, radius: int = 2, n_bits: int = 1024) -> np.ndarray:
    """Circular (Morgan/ECFP-like) fingerprint: hashed atom environments up to ``radius`` bonds, as 0/1 bits."""
    import zlib
    nbr = {i: [] for i in range(len(atoms))}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    ids = [zlib.crc32(repr((el, ar, len(nbr[k]))).encode()) for k, (el, ar) in enumerate(atoms)]
    bits = np.zeros(n_bits, dtype=np.int8)
    for _ in range(radius + 1):
        for v in ids:
            bits[v % n_bits] = 1
        ids = [zlib.crc32(repr((ids[k], sorted((o, ids[j]) for j, o in nbr[k]))).encode()) for k in range(len(atoms))]
    return bits


def tanimoto(a, b) -> float:
    """Tanimoto (Jaccard) similarity of two binary fingerprints: |a ∧ b| / |a ∨ b|."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = np.sum(a | b)
    return float(np.sum(a & b) / union) if union else 0.0


def assemble_molecule(scaffold: str, substituents: dict):
    """Molecular graph (atoms, bonds) of ``scaffold`` carrying ``substituents`` = {site index: substituent name}."""
    atoms, bonds, sites = MOLECULE_SCAFFOLDS[scaffold]
    atoms, bonds = list(atoms), list(bonds)
    for s, name in sorted(substituents.items()):
        f_atoms, f_bonds = MOLECULE_SUBSTITUENTS[name]
        off = len(atoms)
        atoms += f_atoms
        bonds += [(i + off, j + off, o) for i, j, o in f_bonds] + [(sites[s], off, 1)]
    return atoms, bonds


def simulate_molecules(rng: np.random.Generator, n_molecules: int = 2000, noise: float = 0.3) -> dict:
    """Simulate a small-molecule library with a bioactivity (pIC50) and a solubility (logS) assay.

    Each molecule = one of 10 ring scaffolds + 1–3 substituents at the scaffold's attachment
    sites.  Hidden structure–activity rules (the "biology" a model must learn): each scaffold
    has its own baseline potency; a hydrogen-bond donor at site 0 (the key pocket) adds +1.2;
    small lipophilic groups at site 1 add +0.5; bulky groups at site 2 clash (−1.0); very
    lipophilic molecules lose potency (−0.4 per logP unit above 3); plus small random
    site × substituent effects.  Solubility falls with logP and size.

    Returns a dict with ``table`` (SMILES, scaffold, substituents, descriptors, measured
    ``pIC50`` and ``logS``), ``graphs`` (list of (atoms, bonds)), ``scaffold_effect``, and
    ``oracle(scaffold, substituents)`` → noise-free (pIC50, logS) of *any* design, e.g.
    ``oracle("pyridine", {0: "amino", 2: "methyl"})`` — the ground truth for design (Module 39).
    """
    scaffolds, subs = list(MOLECULE_SCAFFOLDS), list(MOLECULE_SUBSTITUENTS)
    weights = rng.dirichlet(np.full(len(scaffolds), 2.0))                 # some scaffolds are rare
    scaffold_effect = dict(zip(scaffolds, rng.normal(5.5, 0.8, len(scaffolds))))
    site_effect = rng.normal(0, 0.2, (4, len(subs)))
    donors = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}
    lipophilic = {"methyl", "ethyl", "chloro", "trifluoromethyl"}
    bulky = {"phenyl", "sulfonamide", "dimethylamino", "trifluoromethyl"}

    def oracle(scaffold: str, substituents: dict):
        """Noise-free (pIC50, logS) of a scaffold + {site: substituent} design under the hidden rules."""
        desc = molecule_descriptors(*assemble_molecule(scaffold, substituents))
        act = scaffold_effect[scaffold] + sum(site_effect[s, subs.index(nm)] for s, nm in substituents.items())
        act += 1.2 * (substituents.get(0) in donors) + 0.5 * (substituents.get(1) in lipophilic) \
            - 1.0 * (substituents.get(2) in bulky) - 0.4 * max(0.0, desc["logp"] - 3.0)
        logs = 0.5 - 0.8 * desc["logp"] - 0.006 * (desc["mol_weight"] - 150) + 0.2 * desc["hbd"]
        return float(act), float(logs)

    rows, graphs, seen = [], [], set()
    while len(rows) < n_molecules:
        sc = scaffolds[rng.choice(len(scaffolds), p=weights)]
        n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
        k = int(rng.integers(1, min(3, n_sites) + 1))
        chosen = {int(s): subs[rng.integers(len(subs))] for s in sorted(rng.choice(n_sites, k, replace=False))}
        key = (sc, tuple(sorted(chosen.items())))
        if key in seen:
            continue
        seen.add(key)
        atoms, bonds = assemble_molecule(sc, chosen)
        act, logs = oracle(sc, chosen)
        rows.append(dict(smiles=molecule_to_smiles(atoms, bonds), scaffold=sc,
                         substituents="; ".join(f"site{s}:{nm}" for s, nm in sorted(chosen.items())),
                         **molecule_descriptors(atoms, bonds),
                         pIC50=act + rng.normal(0, noise), logS=logs + rng.normal(0, noise)))
        graphs.append((atoms, bonds))
    return dict(table=pd.DataFrame(rows), graphs=graphs, scaffold_effect=scaffold_effect, oracle=oracle)


print("🧰 Toolkit loaded:", ", ".join(['simulate_helix_bundle', 'potts_score', 'molecule_to_smiles', 'molecule_descriptors', 'molecular_fingerprint', 'tanimoto', 'assemble_molecule', 'simulate_molecules']))

---
## 1 · Design as an inverse problem

> 📍 **Where we are.** Every earlier module *predicted*: given a sequence or molecule, estimate a property. Design runs
> the arrow backwards — given a target property, produce an object that has it. This section sets up the vocabulary and the
> sheer scale of the problem.

**Forward vs inverse.** A **forward model** maps an object to a property, $f:\text{sequence}\to\text{fitness}$
(Modules 36–37, 40). **Design** seeks the inverse: $\text{property}\to\text{object}$. The inverse is not a function —
many sequences can hit a target, and most sequences hit nothing — so design is really *search*: explore the space to find
objects an (imperfect) forward model, or a real assay, rates highly.

**The design–build–test–learn (DBTL) loop.** Real campaigns iterate: **design** candidates with a model, **build**
(synthesise) a batch, **test** them in an assay, and **learn** — update the model with the new data — then repeat. Because
"test" is slow and expensive, the whole game is to spend each batch well. That is the Bayesian-optimisation problem of
[Module 29](29_gaussian_processes_bayesian_optimization.ipynb), now over sequences and molecules instead of temperatures
and pH.

**The spaces are unimaginably large.** The one number to keep in mind: you can never enumerate. Let us make it concrete.

In [ ]:
# ── 1.1 How big are design spaces? ──
for Lp in [10, 50, 100, 300]:
    print(f"  protein of length {Lp:3d}: 20^{Lp} = 10^{Lp * np.log10(20):.0f} possible sequences")
print(f"\n  drug-like small molecules (common estimate): ~10^60")
print(f"  atoms in the observable universe (for scale): ~10^80")
print(f"  a 4-site library (Module 29): 20^4 = {20**4:,} — the rare case small enough to enumerate")
print("\nEverything larger than a few mutations must be *searched*, guided by a model we only partly trust.")

🔍 **What just happened?** A protein of only 100 residues already has more sequences ($10^{130}$) than there are atoms in
the observable universe ($10^{80}$). No screen, no robot, no amount of compute enumerates this. Design is therefore a
search problem under a budget, and its quality is decided by two things: how cleverly we move through the space, and how
honestly our guiding model reports what it does and does not know. The rest of the module builds both — on spaces small
enough to hold the ground truth, so we can check every claim.

> 🔗 **Builds on.** Searching an intractable space by proposing and accepting moves is [Module 21](21_bayesian_learning_mcmc.ipynb)
> (MCMC and simulated annealing); choosing the next batch under uncertainty is [Module 29](29_gaussian_processes_bayesian_optimization.ipynb)
> (Bayesian optimisation); the surrogate we optimise is a supervised model from [Module 22](22_neural_networks.ipynb)/[40](40_molecular_ml_drug_discovery.ipynb).

---
## 2 · Designing by sampling from an evolutionary model

> 📍 **Where we are.** The gentlest way to write a new protein is to learn the statistics of a natural family and sample
> from them — the sequences a model of evolution finds plausible. We use the Potts model of [Module 37](37_protein_sequences_language_models.ipynb):
> a distribution $P(x)\propto\exp\!\big(\sum_i h_i(x_i)+\sum_{i<j}J_{ij}(x_i,x_j)\big)$ over sequences, with conserved
> positions ($h$) and co-varying contacts ($J$).

🔬 **Assay box — a protein family and its oracle.** We take the contact map of an idealised four-helix bundle
(`simulate_helix_bundle`) and sample a family of 2,000 aligned sequences from a Potts model whose couplings follow three
shared physico-chemical rules — **hydrophobic packing**, **salt bridges**, **size complementarity** (the structured sampler
from [Module 24](24_attention_transformers.ipynb); the toolkit's `simulate_protein_family` uses *random* couplings instead,
which a small neural model in Section 5 could not learn). Because we keep the true $h$ and $J$, `potts_score` is a
**perfect oracle**: the exact "fitness" of any sequence we design. The fittest sampled sequence is our **wild type**.

In [ ]:
# ── 2.1 Three physico-chemical rules for residues in contact (the shared couplings) ──
# Values in the order of AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
HYDROPATHY = np.array([1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8,
                       1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3])   # Kyte–Doolittle; + = oily
CHARGE = np.array([0, 0, -1, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], float)   # D,E −1; K,R +1
VOLUME = np.array([88.6, 108.5, 111.1, 138.4, 189.9, 60.1, 153.2, 166.7, 168.6, 166.7,
                   162.9, 114.1, 112.7, 143.8, 173.4, 89.0, 116.1, 140.0, 227.8, 193.6])    # side-chain Å³ (approx.)


def chem_zscore(v):
    """Centre and scale a property vector (local helper; not the toolkit's `standardize`)."""
    return (v - v.mean()) / v.std()


def chemistry_matrices():
    """Three 20×20 residue–residue preference matrices (offsets removed, largest entry ±1)."""
    raw = {"hydrophobic packing": np.outer(chem_zscore(HYDROPATHY), chem_zscore(HYDROPATHY)),
           "salt bridge": -np.outer(CHARGE, CHARGE),
           "size complementarity": -np.outer(chem_zscore(VOLUME), chem_zscore(VOLUME))}
    out = {}
    for name, M in raw.items():
        M = M - M.mean(0, keepdims=True) - M.mean(1, keepdims=True) + M.mean()
        out[name] = M / np.abs(M).max()
    return out


CHEM = chemistry_matrices()
print("Three chemistry matrices defined — every contact's coupling J_ij is a mix of these plus a little noise.")

In [ ]:
# ── 2.2 Sample a protein family from the Potts model (Gibbs sampling; ≈ 3 s) ──
def sample_chemistry_family(rng, contacts, n_seqs=2000, n_sweeps=60, strength=2.0, noise=0.1):
    """Simulate an MSA from a Potts model whose couplings follow the three chemistry matrices.
    Returns msa (n_seqs × L ints), h (L×20), J (L×L×20×20) and the wild type (fittest sampled sequence)."""
    L, q = contacts.shape[0], 20
    mats = np.stack(list(CHEM.values()))                                    # 3 × 20 × 20
    h = rng.normal(0, 1, (L, q)) * rng.uniform(0.3, 1.5, L)[:, None]        # stronger fields = more conserved
    J = np.zeros((L, L, q, q))
    for i, j in zip(*np.nonzero(np.triu(contacts, 1))):
        w = rng.dirichlet([0.5, 0.5, 0.5]) * rng.uniform(0.5, 1.5)          # which rules this contact uses
        J[i, j] = strength * np.tensordot(w, mats, axes=1) + rng.normal(0, noise, (q, q))
        J[j, i] = J[i, j].T
    nbrs = [np.flatnonzero(contacts[i]) for i in range(L)]
    x, rows = rng.integers(0, q, (n_seqs, L)), np.arange(n_seqs)
    for _ in range(n_sweeps):                                               # Gibbs sampling (Module 21)
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in nbrs[i]:
                logits += J[j, i][x[:, j]]
            g = -np.log(-np.log(rng.random((n_seqs, q))))                   # Gumbel-max = sampling from softmax
            x[rows, i] = np.argmax(logits + g, axis=1)
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, wild_type=wild_type)


def seq_to_str(seq):
    """Integer-coded sequence → amino-acid string."""
    return "".join(AMINO_ACIDS[int(a)] for a in seq)


rng = np.random.default_rng(SEED)
contacts = simulate_helix_bundle()["contacts"]
t0 = time.time()
family = sample_chemistry_family(rng, contacts, n_seqs=2000, n_sweeps=60)
MSA, h_true, J_true, wild_type = family["msa"], family["h"], family["J"], family["wild_type"]
L = len(wild_type)
oracle_fitness = lambda seqs: potts_score(np.atleast_2d(seqs), h_true, J_true)   # the ground-truth protein oracle
print(f"SIMULATED family: {MSA.shape[0]} sequences × {L} positions in {time.time() - t0:.1f} s")
print("wild type:", seq_to_str(wild_type), "| oracle fitness", round(float(oracle_fitness(wild_type)[0]), 1))

In [ ]:
# ── 2.3 🧪 Ground-truth check: sample new sequences at temperature T, measure fitness/diversity/novelty ──
def mcmc_design(rng, T, n=60, steps=400):
    """Metropolis MCMC from the wild type at temperature T (single-residue proposals). Higher T explores further."""
    x = np.tile(wild_type, (n, 1)); cur = oracle_fitness(x); rows = np.arange(n)
    for _ in range(steps):
        pos = rng.integers(0, L, n); aa = rng.integers(0, 20, n)
        xp = x.copy(); xp[rows, pos] = aa; ep = oracle_fitness(xp)
        accept = rng.random(n) < np.exp(np.minimum(0, (ep - cur) / T))      # Metropolis rule (Module 21)
        x[accept], cur[accept] = xp[accept], ep[accept]
    return x


nat_fit = oracle_fitness(MSA)
rows = []
for T in [0.5, 1.0, 2.0, 4.0, 8.0]:
    S = mcmc_design(np.random.default_rng(SEED + 1), T)
    fit = oracle_fitness(S)
    diversity = np.mean([(S[a] != S[b]).sum() for a in range(len(S)) for b in range(a + 1, len(S))])
    novelty = np.mean([(s != MSA[:1600]).sum(1).min() for s in S])          # mutations from nearest of the 1600 natural seqs
    rows.append((T, fit.mean(), diversity, novelty))
samp = pd.DataFrame(rows, columns=["T", "mean_fitness", "diversity", "novelty"]).round(1)
print(samp.to_string(index=False))
print(f"\nfor reference: wild-type fitness {float(oracle_fitness(wild_type)[0]):.1f}, natural-family mean {nat_fit.mean():.1f}")

In [ ]:
# ── 2.4 The fitness–diversity trade-off ──
fig, ax = plt.subplots(figsize=(5.6, 3.8))
sc = ax.scatter(samp.diversity, samp.mean_fitness, c=samp["T"], cmap="plasma", s=90, zorder=3, edgecolor="k")
for _, r in samp.iterrows():
    ax.annotate(f"T={r.T}", (r.diversity, r.mean_fitness), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.axhline(float(oracle_fitness(wild_type)[0]), ls="--", color="0.4", lw=1, label="wild type")
ax.axhline(nat_fit.mean(), ls=":", color="tab:green", lw=1, label="natural-family mean")
ax.set_xlabel("diversity (mean pairwise Hamming distance)"); ax.set_ylabel("mean oracle fitness")
ax.set_title("SIMULATED design by sampling: the fitness–diversity trade-off")
plt.colorbar(sc, ax=ax, label="temperature T"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

🔍 **What just happened?** Sampling from the evolutionary model with a temperature knob traces a clean **trade-off**. At
low $T$ the chains stay close to the wild type and to *each other*: high fitness but low diversity — a batch of near-copies.
As $T$ rises the sampler strays further, fitness falls while diversity and novelty both climb — bolder, more varied designs
that are less likely to work. This is exactly the behaviour Russ *et al.* (2020) exploited to design functional
chorismate-mutase enzymes by sampling a DCA/Potts model: pick a temperature that buys **diversity you can afford at a
fitness you still trust**. At the lowest temperatures the chains even settle *above* the wild type — modest improvements in
its immediate neighbourhood (recall the wild type was only the fittest sequence the family *sampler* happened to draw, not
a true optimum).

⚠️ **Sampling trusts the model's probabilities.** Here the model *is* the truth (we sample the same Potts model we score
with), so high-probability means high-fitness by construction. With a *learned* model fitted to real sequences, the two
diverge — the model may put probability where fitness is low. Sections 4–5 confront that gap.

✏️ **Your turn.** Add a column to 2.3 reporting how often a sampled design beats the wild type (`(fit > wt_fit).mean()`).
At which temperature are improvements over the wild type most common, and why does that fraction collapse as $T$ rises?

### 📄 Research connection — Generative protein design

- **What the field does.** Physics-based design (Rosetta) built novel folds and binders from scratch (Huang, Boyken & Baker 2016; Anishchenko
*et al.* 2021 "de novo proteins by hallucination"). Deep learning reshaped the field: **ProteinMPNN** writes a sequence
for a given backbone (Dauparas *et al.* 2022); **RFdiffusion** generates new backbones by denoising (Watson *et al.*
2023); protein **language models** generate functional sequences (Madani *et al.* 2023 ProGen; Hayes *et al.* 2025
ESM3); and evolution-based models design working enzymes by sampling a learned family distribution (Russ *et al.*
2020).
- **What this notebook builds.** Section 2 samples new sequences from a Potts family at a temperature — the evolution-based design idea in miniature —
and scores them with the exact oracle. Sections 3–5 add optimisation, surrogate pitfalls and inverse folding, the
conceptual cores of the production tools above.
- **Where the frontier is.** As of 2025, the frontier combines diffusion over structure with language-model or MPNN sequence design and in-silico
filtering (AlphaFold self-consistency), extends to binders, enzymes and multi-state/conditional design, and pushes
experimental success rates up. Open problems: designing *function* (catalysis, allostery, dynamics) rather than just a
stable fold, and honest in-silico evaluation that predicts wet-lab success.
- **Read next.** Dauparas *et al.* 2022 (*Science*), *Robust deep learning-based protein sequence design using ProteinMPNN*; Watson
*et al.* 2023 (*Nature*), *De novo design of protein structure and function with RFdiffusion*; Russ *et al.* 2020
(*Science*), *An evolution-based model for designing chorismate mutase enzymes*.

---
## 3 · Optimisation with an oracle: annealing and evolution

> 📍 **Where we are.** Sampling spreads designs across a fitness–diversity curve. If instead we want the **single best**
> sequence an oracle can give us, we optimise. Two classics: **simulated annealing** (MCMC that cools, Module 21) and a
> **genetic algorithm** (mutation, crossover, selection — directed evolution in code). Here the "oracle" is `potts_score`;
> pretend each call is an expensive assay and count them.

In [ ]:
# ── 3.1 Simulated annealing and a genetic algorithm ──
def simulated_annealing(rng, steps=3000, n_restarts=8, T0=3.0, T1=0.05):
    """Metropolis MCMC with a geometric cooling schedule (Module 21). Returns best-fitness trace and best sequence."""
    x = rng.integers(0, 20, (n_restarts, L)); cur = oracle_fitness(x); rows = np.arange(n_restarts)
    best, best_seq = cur.copy(), x.copy(); trace = []
    Ts = T0 * (T1 / T0) ** (np.arange(steps) / steps)
    for t in range(steps):
        pos = rng.integers(0, L, n_restarts); aa = rng.integers(0, 20, n_restarts)
        xp = x.copy(); xp[rows, pos] = aa; ep = oracle_fitness(xp)
        acc = rng.random(n_restarts) < np.exp(np.minimum(0, (ep - cur) / Ts[t]))
        x[acc], cur[acc] = xp[acc], ep[acc]
        improved = cur > best; best[improved], best_seq[improved] = cur[improved], x[improved]
        trace.append(best.max())
    return np.array(trace), best_seq[np.argmax(best)], steps * n_restarts


def genetic_algorithm(rng, gens=120, pop=96, elite=24, mut_rate=1.0):
    """Mutation + one-point crossover + truncation selection. Returns best-fitness trace, best sequence, #oracle calls."""
    P = rng.integers(0, 20, (pop, L)); trace, calls = [], 0
    for _ in range(gens):
        fit = oracle_fitness(P); calls += pop; order = np.argsort(fit)[::-1]
        parents = P[order[:elite]]; trace.append(fit.max())
        children = [P[order[0]]]                                            # keep the champion
        while len(children) < pop:
            a, b = parents[rng.integers(elite)], parents[rng.integers(elite)]
            cut = rng.integers(1, L); child = np.concatenate([a[:cut], b[cut:]])   # crossover
            n_mut = rng.poisson(mut_rate)                                   # mutation
            for _ in range(n_mut):
                child[rng.integers(L)] = rng.integers(20)
            children.append(child)
        P = np.array(children)
    fit = oracle_fitness(P); calls += pop
    return np.array(trace + [fit.max()]), P[np.argmax(fit)], calls


t0 = time.time()
sa_trace, sa_best, sa_calls = simulated_annealing(np.random.default_rng(SEED + 2))
ga_trace, ga_best, ga_calls = genetic_algorithm(np.random.default_rng(SEED + 3))
wt_fit = float(oracle_fitness(wild_type)[0])
print(f"simulated annealing: best fitness {float(oracle_fitness(sa_best)[0]):.1f} in {sa_calls:,} oracle calls")
print(f"genetic algorithm  : best fitness {float(oracle_fitness(ga_best)[0]):.1f} in {ga_calls:,} oracle calls")
print(f"wild type (target to beat): {wt_fit:.1f}   ({time.time() - t0:.0f} s)")

In [ ]:
# ── 3.2 Efficiency per oracle call: annealing vs evolution vs random ──
rng = np.random.default_rng(SEED + 4)
rand_best = np.maximum.accumulate([oracle_fitness(rng.integers(0, 20, (96, L))).max() for _ in range(60)])

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(np.arange(len(sa_trace)) * 8, sa_trace, label=f"simulated annealing (best {sa_trace[-1]:.0f})", lw=1.4)
ax.plot(np.arange(len(ga_trace)) * 96, ga_trace, label=f"genetic algorithm (best {ga_trace[-1]:.0f})", lw=1.4)
ax.plot(np.arange(len(rand_best)) * 96, rand_best, label=f"random search (best {rand_best[-1]:.0f})", lw=1.4)
ax.axhline(wt_fit, ls="--", color="k", lw=1, label=f"wild type ({wt_fit:.0f})")
ax.set_xlabel("oracle calls (assays)"); ax.set_ylabel("best fitness found")
ax.set_title("Optimising a protein against the oracle (SIMULATED)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Hamming distance between the SA and GA winning designs:", int((sa_best != ga_best).sum()),
      "of", L, "positions — the two optimisers found different high-fitness sequences.")

🔍 **What just happened?** Both optimisers blow past random search and past the wild type: with an oracle to query, you do
not need evolution's eons. **Simulated annealing** anneals a single-mutation random walk — hot early (explore), cold late
(exploit) — and, given a large budget, crept to the highest fitness of all. The **genetic algorithm** recombines good
partial solutions by crossover, which helps when fitness is built from semi-independent blocks (as a contact graph is);
here it was the more **call-efficient**, reaching a wild-type-beating design in far fewer oracle calls, though it paid by
evaluating a whole population each generation. Which wins depends on the landscape and on how you count cost (total calls
vs rounds vs final ceiling); here both reach high-fitness designs well above the wild type.

⚠️ **But this used a *perfect* oracle.** We called `potts_score` — the true fitness — thousands of times for free. Real
assays cost days and dollars, so thousands of calls is a fantasy. The realistic move is to *learn* a cheap surrogate from a
few hundred measurements and optimise *that*. Which is exactly where design gets dangerous.

✏️ **Your turn.** Set the GA's `mut_rate` to 0.5 and to 5.0. Too low and it stops exploring; too high and good solutions
are destroyed faster than selection preserves them. Find the mutation rate that reaches the highest fitness in 60
generations — the "error-threshold" intuition from molecular evolution.

---
## 4 · Model-based optimisation and why it drifts out of distribution

> 📍 **Where we are.** We cannot afford thousands of real assays, so we train a **surrogate** on a few hundred measured
> variants and optimise the surrogate. This is **model-based optimisation (MBO)** — and its central failure is the most
> important lesson in the module: *optimising a model pushes designs into regions where the model is wrong, and the model
> rewards itself for going there.*

🔬 **Assay box — a realistic measurement (global epistasis).** Real assays rarely read out the latent biophysical fitness
linearly. A "fraction folded / fraction active" readout **saturates**: once a protein is stable, extra stabilising
mutations barely move the number. We model this with a sigmoid of the Potts fitness,
$y(x)=\sigma\!\big((E(x)-m)/s\big)$ — a form of **global epistasis** (Otwinowski *et al.* 2018). We start not from the
wild type but from a **lead** variant of middling activity (the realistic case: you have a hit, not the optimum), measure
~600 nearby variants with a little noise, and train a surrogate to predict the assay.

In [ ]:
# ── 4.1 A saturating assay, a lead variant, and 600 measured neighbours ──
natE = oracle_fitness(MSA)
E_mid, E_s = float(np.median(natE)), float(natE.std())
def measured_fitness(seqs):
    """Saturating assay readout (fraction folded/active): a sigmoid of the latent Potts fitness. In [0, 1]."""
    return 1.0 / (1.0 + np.exp(-(oracle_fitness(seqs) - E_mid) / E_s))

lead = MSA[np.argsort(natE)[len(natE) // 3]].copy()            # a middling-activity starting point
rng = np.random.default_rng(SEED + 5)
N_TRAIN = 600
train = np.tile(lead, (N_TRAIN, 1))
for r in range(N_TRAIN):                                       # random 1–4 mutations away from the lead
    k = rng.integers(1, 5); pos = rng.choice(L, k, replace=False); train[r, pos] = rng.integers(0, 20, k)
y_train = measured_fitness(train) + rng.normal(0, 0.03, N_TRAIN)
print(f"lead variant: measured {float(measured_fitness(lead)[0]):.3f}, {int((lead != wild_type).sum())} mutations from wild type")
print(f"training set: {N_TRAIN} variants, ≤4 mutations from the lead; measured range "
      f"{y_train.min():.2f}–{y_train.max():.2f}")

In [ ]:
# ── 4.2 Train a surrogate (ridge on one-hot) and optimise it greedily (no guard rails) ──
def one_hot(seqs):
    """Flatten sequences to L×20 one-hot features (float32)."""
    seqs = np.atleast_2d(seqs); O = np.zeros((len(seqs), L * 20), np.float32)
    O[np.arange(len(seqs))[:, None], seqs + np.arange(L) * 20] = 1.0
    return O

surrogate = Ridge(alpha=0.5).fit(one_hot(train), y_train)
surr_pred = lambda seqs: surrogate.predict(one_hot(seqs))

def greedy_optimise(predict, start, radius=None, steps=120, return_path=False):
    """Hill-climb: at each step take the single mutation the surrogate likes best (optionally within a trust region).
    With return_path=True, also return the whole trajectory of sequences (ham 0, 1, 2, … from the start)."""
    x = start.copy(); path = [x.copy()]
    for _ in range(steps):
        C = np.repeat(x[None], L * 20, 0); idx = np.arange(L * 20); C[idx, idx // 20] = idx % 20
        if radius is not None:
            C = C[(C != lead).sum(1) <= radius]
            if len(C) == 0:
                break
        pv = predict(C)
        if pv.max() <= predict(x[None])[0]:
            break
        x = C[int(np.argmax(pv))]; path.append(x.copy())
    return (x, np.array(path)) if return_path else x

design_naive, naive_path = greedy_optimise(surr_pred, lead.copy(), return_path=True)
print("NAIVE MBO design (optimise the surrogate freely):")
print(f"  surrogate predicts {float(surr_pred(design_naive)[0]):.2f}   |   TRUE measured {float(measured_fitness(design_naive)[0]):.3f}"
      f"   |   {int((design_naive != lead).sum())} mutations from the lead")
print("  (a fraction-folded can only be ≤ 1.0 — a prediction above 1 is a red flag that we have left the data.)")

In [ ]:
# ── 4.3 🧪 Ground-truth check: surrogate vs reality as designs drift from the data ──
rng = np.random.default_rng(SEED + 6)
cloud = [naive_path]                                           # the optimisation trajectory (ham 0,1,2,… from the lead)
for _ in range(3000):                                          # plus a cloud of designs at many distances from the lead
    k = rng.integers(1, 30); x = lead.copy(); p = rng.choice(L, k, replace=False); x[p] = rng.integers(0, 20, k)
    cloud.append(x[None])
pool = np.concatenate(cloud, 0)
pred = surr_pred(pool); true = measured_fitness(pool); dist = (pool != lead).sum(1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
sc = a1.scatter(true, pred, c=dist, cmap="viridis", s=10, alpha=0.6)
lims = [min(true.min(), 0) - 0.05, max(pred.max(), 1) + 0.05]
a1.plot(lims, lims, "k--", lw=0.8, label="perfect"); a1.axhline(1.0, color="tab:red", lw=0.8, ls=":", label="assay ceiling = 1")
a1.set_xlabel("TRUE measured fitness"); a1.set_ylabel("surrogate prediction")
a1.set_title("The surrogate inflates as designs drift away"); a1.legend(fontsize=7)
plt.colorbar(sc, ax=a1, label="mutations from lead")
# Right: follow the OPTIMISATION trajectory step by step (this is where optimisation actually goes).
steps_ham = (naive_path != lead).sum(1)
a2.plot(steps_ham, surr_pred(naive_path), "-o", color="tab:orange", ms=3, label="surrogate prediction")
a2.plot(steps_ham, measured_fitness(naive_path), "-s", color="tab:blue", ms=3, label="TRUE measured")
a2.axhline(1.0, color="tab:red", lw=0.8, ls=":", label="assay ceiling = 1")
a2.set_xlabel("mutations from the lead (greedy optimisation step)"); a2.set_ylabel("measured fitness")
a2.set_title("Along the optimisation path: predicted soars, true peaks then dips"); a2.legend(fontsize=7.5)
plt.tight_layout(); plt.show()
print(f"correlation(prediction, truth) over the whole cloud: {np.corrcoef(pred, true)[0, 1]:.2f}  "
      f"— but it breaks down along the optimisation path, exactly where the search goes.")

🔍 **What just happened? The defining failure of model-based design.** The surrogate is accurate *near its training data*
(small Hamming distance from the lead), but as greedy optimisation stacks "beneficial" mutations the prediction climbs
without bound — past the physical ceiling of 1.0 — while the **true** measured fitness saturates and then falls. The
additive surrogate extrapolates each mutation's local effect as if they simply add; reality has **global epistasis** (the
sigmoid) and **interactions** (the couplings), so the stacked design is nothing like what the model promises. Optimising
the model did not find a great protein; it found an **adversarial example of our own ignorance** (Brookes *et al.* 2019;
Fannjiang & Listgarten 2020). The predicted-vs-true scatter fans off the diagonal precisely in the direction optimisation
pushes.

This is the same danger flagged in [Module 29 §8](29_gaussian_processes_bayesian_optimization.ipynb): acquisition that
exploits surrogate errors out of distribution. Two fixes, both about **staying honest about what the model knows**.

In [ ]:
# ── 4.4 🧪 Fix 1 — trust region; Fix 2 — pessimism with a Gaussian-process lower bound ──
# Fix 1: optimise the same surrogate but forbid straying more than `radius` mutations from the lead.
design_tr = greedy_optimise(surr_pred, lead.copy(), radius=6)

# Fix 2: a GP surrogate (Hamming kernel, Module 29) whose uncertainty GROWS out of distribution; optimise a
# pessimistic lower confidence bound (mean − β·sd), so designs the GP is unsure about are penalised.
Otr = one_hot(train)
def gp_mean_sd(cand, gamma=3.0, sig=0.35, noise=0.05):
    """Posterior mean and sd of a Hamming-kernel GP: k(x,x') = sig²·exp(−γ·Hamming/L)."""
    Oc = one_hot(cand)
    Ktt = sig**2 * np.exp(-gamma * (L - Otr @ Otr.T) / L) + (noise**2 + 1e-6) * np.eye(len(train))
    Lc = cholesky(Ktt, lower=True); alpha = cho_solve((Lc, True), y_train - y_train.mean())
    Kct = sig**2 * np.exp(-gamma * (L - Oc @ Otr.T) / L)
    mean = Kct @ alpha + y_train.mean()
    v = solve_triangular(Lc, Kct.T, lower=True); sd = np.sqrt(np.maximum(sig**2 - np.sum(v**2, 0), 1e-9))
    return mean, sd

gp_m, gp_s = gp_mean_sd(pool)
design_pess = pool[int(np.argmax(gp_m - 1.0 * gp_s))]          # maximise the pessimistic lower bound over the pool

print(f"{'strategy':22s} {'predicted':>10s} {'TRUE':>8s} {'mutations from lead':>20s}")
for name, d, pv in [("naive (free)", design_naive, float(surr_pred(design_naive)[0])),
                    ("trust region (≤6)", design_tr, float(surr_pred(design_tr)[0])),
                    ("GP pessimism (m−σ)", design_pess, float(gp_mean_sd(design_pess[None])[0][0]))]:
    print(f"{name:22s} {pv:10.2f} {float(measured_fitness(d)[0]):8.3f} {int((d != lead).sum()):20d}")
print(f"\nlead measured {float(measured_fitness(lead)[0]):.3f} | corr(GP sd, distance) = {np.corrcoef(gp_s, dist)[0,1]:.2f} "
      f"(the GP knows it is extrapolating)")

🔍 **What just happened?** The headline is the naive design's number: the surrogate predicts a "fraction folded" of about
1.5 — a physical impossibility — while the true assay is roughly a third of that. You would have synthesised a design the
model raved about and reality shrugs at. The trajectory panel shows why: as greedy optimisation stacks "beneficial"
mutations the prediction climbs without bound, but true fitness peaks partway out and then **declines** — optimising the
model walked us off a cliff the model could not see (global epistasis plus interactions; Brookes *et al.* 2019; Fannjiang &
Listgarten 2020).

Both fixes keep the search **honest**. The **trust region** forbids straying far from the measured data, so the surrogate is
only trusted where it is still informative, and its design is a real (if modest) improvement over the lead. **Pessimism**
uses a Gaussian process (the Hamming-kernel GP of [Module 29](29_gaussian_processes_bayesian_optimization.ipynb)) whose
uncertainty *grows* with distance from the data (correlation of GP uncertainty with distance is high); optimising the lower
confidence bound $\mu-\sigma$ automatically backs away from the over-confident corner, and here it gave the best of the
three *honest* designs. Note what neither fix did: reach the true peak partway out on the trajectory. From training data
only a few mutations from the lead, that peak is genuinely **unknowable** — no honest model can see it — which is exactly
why real design **closes the loop**: measure the safe design, add it to the data, expand the trust region, and step out
again (the DBTL loop; Trabucco *et al.* 2021; Stanton *et al.* 2022).

The general principle (Trabucco *et al.* 2021; Stanton *et al.* 2022): **optimise a conservative objective, or stay inside
the trust region, and close the loop** — build the design, measure it, add it to the data, and only then step further out.
That is the DBTL loop with its safety catch on.

✏️ **Your turn.** Sweep the trust-region `radius` from 2 to 20 in 4.4 and plot the true measured fitness of the resulting
design against the radius. You should see a sweet spot: too tight and you cannot improve, too loose and you fall off the
distribution — the bias–variance trade-off of [Module 23](23_model_selection.ipynb) in design space.

### 📄 Research connection — Model-based optimisation and its pitfalls

- **What the field does.** Model-based optimisation trains a surrogate on measured designs and optimises it to propose new ones. Its hazard —
optimisation exploits surrogate error out of distribution — is well documented: Brookes *et al.* (2019, CbAS) and
Fannjiang & Listgarten (2020) frame it as design drifting off the data manifold; Trabucco *et al.* (2021, COMs) add
**conservative** objectives that penalise over-confident predictions; Stanton *et al.* (2022) bring Bayesian
optimisation with deep-kernel surrogates to biological sequence design.
- **What this notebook builds.** Section 4 provokes the failure (an additive surrogate on a saturating assay) and fixes it with a trust region and a
pessimistic GP lower confidence bound — the same calibrated-uncertainty idea as Module 29, now defending a design loop.
- **Where the frontier is.** As of 2025, open questions include how conservative to be versus how boldly to explore, trustworthy uncertainty for
large neural surrogates (and protein language models) out of distribution, offline MBO benchmarks (Design-Bench), and
coupling generative models with calibrated uncertainty and active experimentation (the self-driving-lab vision of
Module 29 and 43).
- **Read next.** Brookes, Park & Listgarten 2019 (*ICML*), *Conditioning by adaptive sampling for robust design*; Fannjiang &
Listgarten 2020 (*NeurIPS*), *Autofocused oracles for model-based design*; Trabucco *et al.* 2021 (*ICML*),
*Conservative objective models for effective offline model-based optimization*.

---
## 5 · Inverse folding in miniature

> 📍 **Where we are.** The modern protein-design pipeline is **structure first, sequence second**: pick or generate a
> backbone, then ask "what sequence folds into this shape?" That second step is **inverse folding**, and ProteinMPNN
> (Dauparas *et al.* 2022) is its workhorse. We build a tiny version: predict each residue from the residues of its
> **structural neighbours** — a message-passing idea ([Module 25](25_graph_neural_networks.ipynb)) on the contact graph.

*The model.* For each position $i$, aggregate the one-hot residues of its contacting neighbours (a sum over the contact
graph), concatenate a learned position identity, and predict a distribution over the 20 amino acids with a small shared
MLP. Trained on the family's sequences to recover each residue from its neighbourhood, it learns both the per-position
preferences ($h$) and the pairwise compatibility rules ($J$) — the same information a Potts model encodes, but as an
**amortised** neural network that can score any position of any sequence in one pass.

*In words:* "given who my neighbours are in 3-D, which amino acid belongs here?"

In [ ]:
# ── 5.1 A tiny inverse-folding network (predict a residue from its structural neighbours) ──
NBR = torch.tensor(contacts.astype(np.float32))               # L × L contact (neighbour) mask
POSID = torch.eye(L)

def neighbour_features(X):
    """X: (B, L) ints → (B, L, 20): for each position, the summed one-hot of its contacting neighbours' residues."""
    oh = F.one_hot(X, 20).float()
    return torch.einsum("ij,bjk->bik", NBR, oh)

class InverseFold(nn.Module):
    """Shared MLP: [neighbour residue sums (20) ‖ position identity (L)] → logits over 20 amino acids."""
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20 + L, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 20))

    def forward(self, X):
        B = X.shape[0]
        feats = torch.cat([neighbour_features(X), POSID[None].expand(B, L, L)], dim=-1)
        return self.net(feats)                                 # (B, L, 20)

torch.manual_seed(SEED)
msa_t = torch.tensor(MSA)
train_seqs, test_seqs = msa_t[:1600], msa_t[1600:]
model = InverseFold()
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
t0 = time.time()
for step in range(400):
    batch = train_seqs[torch.randint(0, len(train_seqs), (64,))]
    loss = F.cross_entropy(model(batch).reshape(-1, 20), batch.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters, 400 steps in {time.time() - t0:.0f} s, "
      f"final loss {loss.item():.3f}")

In [ ]:
# ── 5.2 🧪 Ground-truth check: sequence recovery vs a profile baseline ──
model.eval()
with torch.no_grad():
    pred_test = model(test_seqs).argmax(-1)
    recovery = (pred_test == test_seqs).float().mean().item()
    wt_pred = model(torch.tensor(wild_type[None])).argmax(-1)[0].numpy()
    wt_recovery = float((wt_pred == wild_type).mean())
profile = np.array([np.bincount(MSA[:1600, i], minlength=20).argmax() for i in range(L)])   # per-column consensus
profile_recovery = float((profile == test_seqs.numpy()).mean())
print(f"sequence recovery on held-out natural sequences:")
print(f"  structure-aware model : {recovery:.3f}")
print(f"  profile (consensus)   : {profile_recovery:.3f}   (ignores who the neighbours are)")
print(f"  random guess          : {1/20:.3f}")
print(f"\nwild-type recovery: structure-aware {wt_recovery:.3f} vs profile {float((profile == wild_type).mean()):.3f}")

In [ ]:
# ── 5.3 Decode novel designs for the backbone and score them with the oracle ──
def decode(n, sweeps=25, T0=1.0, T1=0.2, seed=0):
    """Generate sequences for the backbone by annealed sequential resampling (ProteinMPNN-style decoding)."""
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(0, 20, (n, L), generator=g)
    for s in range(sweeps):
        T = T0 + (T1 - T0) * s / (sweeps - 1)
        for i in torch.randperm(L, generator=g).tolist():
            with torch.no_grad():
                p = torch.softmax(model(X)[:, i] / T, -1)
            X[:, i] = torch.multinomial(p, 1, generator=g).squeeze(1)
    return X.numpy()

t0 = time.time()
designs = decode(40, seed=SEED)
d_fit = oracle_fitness(designs)
novelty = np.array([(d != MSA[:1600]).sum(1).min() for d in designs])
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.hist(nat_fit, bins=30, alpha=0.5, label="natural family", color="tab:green", density=True)
ax.hist(d_fit, bins=15, alpha=0.6, label="inverse-folding designs", color="tab:purple", density=True)
ax.axvline(wt_fit, ls="--", color="k", lw=1, label="wild type")
ax.set_xlabel("oracle fitness"); ax.set_ylabel("density")
ax.set_title("SIMULATED inverse-folding designs score as foldable as (or better than) natural"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"designs: mean oracle fitness {d_fit.mean():.1f} (natural {nat_fit.mean():.1f}, wild type {wt_fit:.1f}); "
      f"novelty {novelty.mean():.0f} mutations from the nearest natural sequence  ({time.time()-t0:.0f} s)")

🔍 **What just happened?** The structure-aware model recovers the held-out sequences better than a position-wise
consensus (**profile**) baseline and far better than chance, because it uses *who the structural neighbours are* — the
coevolutionary signal a profile throws away. On the wild-type backbone it recovers nearly half the native residues, in
the same spirit (if not the same magnitude) as ProteinMPNN's ~50 % recovery on real proteins. Decoding by **annealed
sequential resampling** — fill positions in a random order, cooling as you go, exactly the ProteinMPNN decoding scheme —
produces sequences that are **novel** (dozens of mutations from any natural sequence) yet score as foldable as the natural
family, often above the wild type. The model has learned the backbone's sequence constraints well enough to *write* new
solutions to them.

⚠️ **Recovery is a proxy, not the goal.** High native-sequence recovery correlates with design success but does not
guarantee the protein folds or functions — a sequence can match natives at easy positions and fail at the hard ones.
Production pipelines therefore *filter* designs by predicting their structure (AlphaFold) and checking it matches the
target backbone (Section 7). Our oracle lets us check directly; real projects cannot, and must triangulate.

✏️ **Your turn.** Lower the decoding `T1` to 0.05 (greedier) and raise it to 0.6 (bolder). How do the designs' mean oracle
fitness and their novelty trade off? You have just rediscovered the sampling-temperature knob of Section 2, now inside a
neural decoder.

---
## 6 · Designing small molecules

> 📍 **Where we are.** The same recipe — surrogate + search — designs drugs, with molecules instead of sequences. We use
> the toolkit's molecule library (from [Module 40](40_molecular_ml_drug_discovery.ipynb)): ring **scaffolds** decorated
> with **substituents**, with a hidden structure–activity rule set and a noise-free **oracle** for true potency (pIC50)
> and solubility (logS).

In [ ]:
# ── 6.1 A molecule library, a random-forest surrogate, and the ground-truth oracle ──
rng = np.random.default_rng(SEED)
mol = simulate_molecules(rng, 2500)                           # SIMULATED library with a noise-free oracle
library, graphs, oracle = mol["table"], mol["graphs"], mol["oracle"]
SCAFFOLDS, SUBSTITUENTS = list(MOLECULE_SCAFFOLDS), list(MOLECULE_SUBSTITUENTS)

X_fp = np.array([molecular_fingerprint(a, b) for a, b in graphs])
y_pic50 = library["pIC50"].to_numpy(); y_logs = library["logS"].to_numpy()
t0 = time.time()
rf_pic50 = RandomForestRegressor(100, max_features=0.3, random_state=SEED, n_jobs=2).fit(X_fp, y_pic50)
rf_logs = RandomForestRegressor(100, max_features=0.3, random_state=SEED, n_jobs=2).fit(X_fp, y_logs)
print(f"library: {len(library)} molecules | fingerprint dim {X_fp.shape[1]} | RF surrogates fit in {time.time()-t0:.0f} s")
print(f"library potency: max measured pIC50 {y_pic50.max():.2f}; surrogate train R² {rf_pic50.score(X_fp, y_pic50):.2f}")

In [ ]:
# ── 6.2 A genetic algorithm over scaffolds and substituents ──
def genome_fingerprint(genome):
    """(scaffold, {site: substituent}) → circular fingerprint of the assembled molecule."""
    return molecular_fingerprint(*assemble_molecule(*genome))

def random_genome(g):
    sc = SCAFFOLDS[g.integers(len(SCAFFOLDS))]; n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
    k = int(g.integers(1, min(3, n_sites) + 1))
    return (sc, {int(s): SUBSTITUENTS[g.integers(len(SUBSTITUENTS))] for s in g.choice(n_sites, k, replace=False)})

def mutate(genome, g):
    sc, sub = genome; sub = dict(sub); r = g.random(); n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
    if r < 0.3:                                               # swap the scaffold (drop sites it no longer has)
        sc = SCAFFOLDS[g.integers(len(SCAFFOLDS))]; n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
        sub = {k: v for k, v in sub.items() if k < n_sites}
    elif r < 0.7 and n_sites:                                 # add/replace a substituent
        sub[int(g.integers(n_sites))] = SUBSTITUENTS[g.integers(len(SUBSTITUENTS))]
    elif sub:                                                 # remove a substituent
        del sub[list(sub)[g.integers(len(sub))]]
    if not sub:
        sub = {0: SUBSTITUENTS[g.integers(len(SUBSTITUENTS))]}
    return (sc, sub)

def design_ga(objective, gens=40, pop=60, elite=20, seed=1):
    """Genetic algorithm over molecular designs. `objective(list_of_genomes) -> scores`. Returns final population."""
    g = np.random.default_rng(seed); P = [random_genome(g) for _ in range(pop)]
    for _ in range(gens):
        order = np.argsort(objective(P))[::-1]; parents = [P[i] for i in order[:elite]]
        P = parents + [mutate(parents[g.integers(elite)], g) for _ in range(pop - elite)]
    return P

predict_pic50 = lambda genomes: rf_pic50.predict(np.array([genome_fingerprint(x) for x in genomes]))
t0 = time.time()
pop = design_ga(predict_pic50, seed=SEED + 1)
best = pop[int(np.argmax(predict_pic50(pop)))]
true_pic50, true_logs = oracle(*best)
print(f"best design: {best[0]} + {best[1]}")
print(f"  SMILES {molecule_to_smiles(*assemble_molecule(*best))}")
print(f"  surrogate pIC50 {float(predict_pic50([best])[0]):.2f}  |  TRUE pIC50 {true_pic50:.2f}  |  TRUE logS {true_logs:.2f}")
print(f"  (library best measured pIC50 {y_pic50.max():.2f})   [{time.time()-t0:.0f} s]")

In [ ]:
# ── 6.3 🧪 Ground-truth check: did the GA recover the hidden structure–activity rules? ──
# The simulator's rules: H-bond donor at site 0 (+1.2), lipophilic at site 1 (+0.5), bulky at site 2 (−1.0).
donors = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}
top = sorted(pop, key=lambda gmn: oracle(*gmn)[0], reverse=True)[:15]
frac_donor0 = np.mean([g[1].get(0) in donors for g in top])
print(f"top-15 designs by TRUE potency:")
print(f"  carry an H-bond donor at site 0 (the +1.2 rule): {frac_donor0:.0%} of them")
preds = predict_pic50(top); trues = np.array([oracle(*g)[0] for g in top])
print(f"  surrogate vs TRUE on these designs: mean predicted {preds.mean():.2f}  vs  mean true {trues.mean():.2f}")
print(f"  (the RF stays fairly calibrated here — chemical space is small; on vast real space the Section-4 caution returns)")

In [ ]:
# ── 6.4 Multi-objective design: the potency–solubility Pareto front ──
# Collect many designs (union of a potency-GA and a solubility-aware GA) and find the non-dominated set.
pop_sol = design_ga(lambda G: predict_pic50(G) + 1.2 * rf_logs.predict(np.array([genome_fingerprint(x) for x in G])),
                    seed=SEED + 2)
allg = {(g[0], tuple(sorted(g[1].items()))): g for g in pop + pop_sol}.values()
cand = list(allg)
P = np.array([oracle(*g) for g in cand])                       # TRUE (pIC50, logS) of every candidate
is_pareto = np.ones(len(P), bool)
for i in range(len(P)):
    is_pareto[i] = not np.any(np.all(P >= P[i], 1) & np.any(P > P[i], 1))

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.scatter(library["pIC50"], library["logS"], s=8, alpha=0.25, color="0.6", label="library (SIMULATED)")
ax.scatter(P[:, 0], P[:, 1], s=16, color="tab:purple", alpha=0.6, label="designs")
front = P[is_pareto][np.argsort(P[is_pareto, 0])]
ax.plot(front[:, 0], front[:, 1], "-o", color="tab:red", ms=4, label="Pareto front (designs)")
ax.set_xlabel("TRUE potency (pIC50) →"); ax.set_ylabel("TRUE solubility (logS) →")
ax.set_title("Multi-objective molecule design: potency vs solubility"); ax.legend(fontsize=7.5)
plt.tight_layout(); plt.show()
print(f"{is_pareto.sum()} designs are Pareto-optimal (no other design beats them on BOTH potency and solubility).")

🔍 **What just happened?** A random-forest surrogate on fingerprints, driven by a genetic algorithm over scaffolds and
substituents, designed molecules the **oracle** confirms are potent — and the top designs rediscovered the simulator's
hidden rule (an H-bond donor at the key site), without ever being told it. Because potency and solubility trade off (large
lipophilic groups raise potency but sink solubility — Lipinski's world, Module 40), there is no single best molecule but a
**Pareto front**: the set of designs none of which can be beaten on *both* objectives at once. A real project chooses along
that front according to its priorities, then makes and tests a handful.

Note the honest footnote in 6.3: here the RF surrogate stays roughly calibrated on the designs, because this chemical space
is small and densely sampled. On real, vast chemical space the Section-4 story returns in full — surrogates extrapolate,
designs drift out of distribution, and you need trust regions, uncertainty and the DBTL loop. Molecule generation also has
a constraint proteins (mostly) do not: a design must be **synthesisable** (Section 9).

✏️ **Your turn.** Change the multi-objective weight (the `1.2` on `rf_logs`) to 0 and to 3. Watch the cloud of designs
slide along the trade-off from potent-but-greasy to soluble-but-weak. Which weight would *you* hand the chemists, and what
else (toxicity, selectivity, synthesis) is missing from this two-objective picture?

### 📄 Research connection — Molecule generation and benchmarks

- **What the field does.** De novo molecule generation spans continuous latent-space models (Gómez-Bombarelli *et al.* 2018, a SMILES VAE),
genetic algorithms over molecular graphs (Jensen 2019), reinforcement-learning generators (REINVENT), and
diffusion/flow models over graphs or 3-D conformers. Because it is easy to report misleading success, the field relies
on standard benchmarks: GuacaMol (Brown *et al.* 2019), MOSES (Polykovskiy *et al.* 2020) and the Practical Molecular
Optimization benchmark (Gao *et al.* 2022), which scores methods by oracle *efficiency*.
- **What this notebook builds.** Section 6 is a graph genetic algorithm with a random-forest surrogate and a ground-truth oracle, plus a Pareto front
for multi-objective design — the same loop as production generators, minus learned priors and synthesis constraints.
- **Where the frontier is.** As of 2025, active directions include generation conditioned on a protein pocket (structure-based design), models that
build in **synthesisability** (generating synthesis routes, not just structures), sample-efficient optimisation under
tight oracle budgets, and — as a caution — evidence that simple baselines rival elaborate generators on honest
benchmarks (Gao *et al.* 2022; Renz *et al.* 2019).
- **Read next.** Gómez-Bombarelli *et al.* 2018 (*ACS Cent Sci*), *Automatic chemical design using a data-driven continuous
representation*; Brown *et al.* 2019 (*J Chem Inf Model*), *GuacaMol: benchmarking models for de novo molecular
design*; Gao *et al.* 2022 (*NeurIPS Datasets & Benchmarks*), *Sample efficiency matters* (the PMO benchmark).

---
## 7 · Modern pipelines (conceptual)

> 📍 **Where we are.** Our four ideas — sampling, optimisation, surrogate MBO, inverse folding — are the building blocks of
> the pipelines in today's papers. Here is how they fit together, in words.

**Proteins: the structure-first stack.** A typical 2023–2025 de novo design run chains three models:

1. **Generate a backbone.** [**RFdiffusion**](https://github.com/RosettaCommons/RFdiffusion) (Watson *et al.* 2023) starts
   from noise and *denoises* into a 3-D backbone, optionally conditioned on a target (a binding site, a symmetry, a motif
   to scaffold). This is the diffusion idea of [Module 28](28_diffusion_flow_matching.ipynb), run over coordinates.
2. **Write a sequence for it.** **ProteinMPNN** (Dauparas *et al.* 2022) — the full version of Section 5 — predicts a
   sequence likely to fold into that backbone.
3. **Filter in silico.** Predict the designed sequence's structure (AlphaFold2 / ESMFold) and keep only designs whose
   predicted structure matches the intended backbone (low RMSD, high confidence) — "**self-consistency**". Only the
   survivors are synthesised and tested.

**Language-model design.** Protein language models (Module 37) can *generate* sequences directly: **ProGen** (Madani
*et al.* 2023) produced functional enzymes; **ESM3** (Hayes *et al.* 2025) generates over sequence, structure and function
jointly. These replace steps 1–2 with a single generative model.

**Evaluation is the hard part.** Designs are judged by **designability** (does a structure predictor agree the sequence
folds to the target?), **diversity** and **novelty** (are they new, not memorised?), and ultimately the **experimental
success rate** (what fraction express, fold and function in the lab — still the only verdict that counts). Reported hit
rates have risen markedly but remain the true benchmark; in-silico metrics are proxies, as Section 5's warning showed.

**Molecules** follow the same shape: a generator (VAE, RL, GA, or diffusion) proposes candidates, surrogate models and
docking score them, filters enforce drug-likeness and synthesisability, and a small set is made and assayed (Module 40).

---
## 8 · Responsible design

> 📍 **Where we are.** Writing biology is powerful, and power cuts both ways. This section treats the **dual-use** question
> at a conceptual level — the responsibilities that come with the tools, not any operational detail.

The same methods that design a beneficial enzyme or a safer drug could, in principle, be turned toward harm — this is the
**dual-use dilemma** of the life sciences, long recognised for biology as a whole and now sharpened by generative models.
A widely discussed illustration: Urbina *et al.* (2022) pointed out that a model trained to predict and *avoid* toxicity
can be run "in reverse" to search for *more* toxic molecules — the optimisation machinery of this very notebook, pointed
the wrong way. They reported the conceptual risk deliberately, without releasing operational details, precisely to prompt
safeguards. The lesson is general: **an optimiser does whatever its objective says, and a predictor of harm is a map of
harm.**

What responsible practice looks like, as of 2025, at a high level:

- **Guard the physical chokepoints.** The step from a digital sequence to a real molecule runs through DNA/peptide
  synthesis providers. Biosecurity increasingly relies on **synthesis screening** (checking orders against sequences of
  concern) and **know-your-customer** checks, so that designing something dangerous on a laptop does not mean it can be
  made. Supporting and improving this screening is a concrete contribution modellers can make.
- **Think before releasing.** Communities have adopted norms for **responsible development and release** of design tools
  and models — risk assessment before publication, staged or controlled access for the highest-risk capabilities, and
  responsible-disclosure channels. The 2024 community commitments on responsible AI protein design are one example of
  researchers setting such norms themselves.
- **Build safety in, and keep humans accountable.** Red-team your own tools, log and review what autonomous
  design–build–test loops (Module 29; agents in [Module 43](43_llms_ai_agents.ipynb)) propose, and keep a human
  responsible for what gets made. Governance, access control and oversight are part of the engineering, not an afterthought.

This notebook stays entirely in a **simulated** world — a toy four-helix bundle and a fictional small-molecule library — for
exactly these reasons: it teaches the methods and their failure modes without providing a recipe for anything real. The
goal of learning generative design is to use it well, and to help build the safeguards that let the field do the same.

---
## 9 · 🧭 From toy to real data

Our oracle was a Potts model and a formula; real design is harder and better-equipped.

**Production tools.** For proteins: [**ProteinMPNN**](https://github.com/dauparas/ProteinMPNN) (inverse folding),
[**RFdiffusion**](https://github.com/RosettaCommons/RFdiffusion) and [**Chroma**](https://github.com/generatebio/chroma)
(backbone generation), [**ColabDesign**](https://github.com/sokrypton/ColabDesign) (hallucination and design with
AlphaFold in the loop), and ESM/ESM3 for language-model design — with structure prediction (AlphaFold2, ESMFold) as the
in-silico filter. For molecules: REINVENT-style generators, graph/3-D diffusion models, and docking for structure-based
scoring, benchmarked on [**GuacaMol**](https://github.com/BenevolentAI/guacamol), **MOSES** and the **PMO** suite.

**Scale and compute.** Backbone diffusion and structure-prediction filtering want a GPU and run for minutes to hours per
design campaign; inverse folding and molecule GAs are cheaper. Designs are generated in the thousands and *filtered* down
to the few you can afford to make.

**Evaluation and data.** Public landscapes to practise surrogates and design on (reading, not downloads):
[**ProteinGym**](https://proteingym.org) and FLIP for protein fitness, [**ChEMBL**](https://www.ebi.ac.uk/chembl/) and the
[**Therapeutics Data Commons**](https://tdcommons.ai) for molecular properties. The [**PDB**](https://www.rcsb.org) and
[**AlphaFold DB**](https://alphafold.ebi.ac.uk) supply the backbones design starts from.

**The first three things that go wrong.** (1) **The oracle is not real** — in-silico scores (even AlphaFold confidence)
are proxies; the only verdict is the wet lab, and designs that look perfect in silico routinely fail to express, fold or
function. (2) **Optimisation exploits the surrogate out of distribution** (Section 4), so trust regions, calibrated
uncertainty and closing the DBTL loop are not optional. (3) **A design must be *makeable*** — a protein must express and be
soluble; a molecule must be **synthesisable** (retrosynthesis/SA scores) — or it is a picture, not a candidate. And
throughout, the responsibilities of Section 8 travel with the capability.

---
## 📌 Key takeaways

- **Design is inverse prediction = search** over spaces too large to enumerate ($20^{100}\approx10^{130}$ sequences;
  $\sim10^{60}$ molecules), guided by an imperfect model inside a **design–build–test–learn** loop.
- **Sampling from an evolutionary model** trades fitness against diversity via a temperature: low $T$ gives safe,
  high-fitness, low-novelty designs; high $T$ gives bold, diverse, lower-fitness ones (the Russ *et al.* 2020 idea).
- **With an oracle, optimisation is easy** — simulated annealing and genetic algorithms beat random search and the wild
  type in thousands of calls — **but a real oracle is too expensive to call thousands of times.**
- **Model-based optimisation's defining failure:** optimise a learned surrogate and designs drift out of distribution,
  where the surrogate's predictions soar (past the physical ceiling) while true fitness falls. **Trust regions** and
  **pessimism** (a GP lower confidence bound whose uncertainty grows out of distribution — Module 29) recover honest,
  useful designs.
- **Inverse folding** predicts a residue from its structural neighbours; our miniature beat a profile baseline on recovery
  and decoded **novel** sequences the oracle scores as foldable — the ProteinMPNN idea in small.
- **Molecule design** = surrogate + genetic algorithm over scaffolds/substituents; the GA rediscovered the hidden SAR
  rule and traced a **potency–solubility Pareto front**; multi-objective trade-offs, not single winners, are the reality.
- **Production pipelines** chain generation → sequence design → in-silico filtering (RFdiffusion → ProteinMPNN →
  AlphaFold), with experimental success rate as the only true metric.
- **Generative design is dual-use.** Safeguards — synthesis screening, responsible release, oversight and human
  accountability — are part of doing it well.

---
## ✅ Checkpoint

1. Why is design called an *inverse* problem, and why is it not simply 'run the forward model backwards'?
   <details><summary>Answer</summary>The forward model maps object → property. Design wants property → object, but that inverse is not a function: many objects give the same property and most give none. So design is a *search* for objects a (forward) model or assay rates highly, not a closed-form inversion.</details>
2. In Section 2, raising the sampling temperature $T$ lowered mean fitness but raised diversity and novelty. In one sentence, why?
   <details><summary>Answer</summary>Higher T flattens the model's distribution, so the sampler accepts lower-probability (lower-fitness) moves and wanders further from the wild type — more diverse and novel sequences, at a cost in fitness. It is the exploration/exploitation trade-off set by temperature.</details>
3. A colleague optimises a surrogate and reports a design the model scores at a 'fraction folded' of 1.4. Why should you be alarmed, and what single number would you check first?
   <details><summary>Answer</summary>A fraction cannot exceed 1 — the surrogate is extrapolating wildly, a hallmark of leaving the training distribution. Check the design's distance (e.g. Hamming/Tanimoto) to the nearest training example; large distance means the prediction is untrustworthy (Section 4).</details>
4. Explain how a **trust region** and a **pessimistic (lower-confidence-bound) objective** each prevent out-of-distribution over-confidence, and how they differ.
   <details><summary>Answer</summary>A trust region hard-limits how far a design may stray from the data, so the surrogate is only used where it is accurate. Pessimism keeps optimising globally but subtracts the model's uncertainty (μ − β·σ); where the model is unsure (out of distribution) the bound collapses, so the optimiser avoids those regions on its own. Trust region = a hard constraint; pessimism = a soft, uncertainty-aware penalty (and needs calibrated uncertainty, e.g. a GP).</details>
5. What does **sequence recovery** measure in inverse folding, and why is high recovery necessary but not sufficient for a successful design?
   <details><summary>Answer</summary>It is the fraction of positions where the model's top amino acid matches a known native sequence for that backbone. High recovery means the model captured the backbone's constraints, but a sequence can match at easy positions and fail at the few that decide folding/function — so designs still need structural filtering and, ultimately, experiment.</details>
6. Why is a **Pareto front** the right answer to 'design the best molecule' when potency and solubility conflict, rather than a single molecule?
   <details><summary>Answer</summary>When objectives trade off, no single molecule is best on all of them; the Pareto front is the set of non-dominated designs (you cannot improve one objective without hurting another). The project then picks a point on the front by its priorities — a decision the data alone cannot make.</details>
7. Code reading: in `greedy_optimise`, `C = C[(C != lead).sum(1) <= radius]` runs only when `radius` is set. What does it enforce, and what happens to the naive run where `radius=None`?
   <details><summary>Answer</summary>It keeps only candidate sequences within `radius` mutations of the lead — the trust region. With `radius=None` the line is skipped and the hill-climb may wander arbitrarily far from the data, straight into the over-confident region of Section 4.</details>
8. A toxicity-prediction model can be run 'in reverse' to propose more toxic molecules. What does this say about objectives, and name one safeguard that does not depend on restricting the model itself.
   <details><summary>Answer</summary>An optimiser pursues whatever objective it is given — a predictor of harm doubles as a map toward harm, so the danger is in the objective, not only the model. A safeguard independent of the model: screening at DNA/compound synthesis providers (plus know-your-customer checks), which guards the physical step from design to reality.</details>

## 🧠 Questions & discussion

1. Model-based optimisation fails by exploiting the surrogate out of distribution. Design a DBTL protocol that spends each round's budget partly on *exploiting* the surrogate's best designs and partly on *testing* its most uncertain confident predictions. How would you split the budget, and how would you know the split was right?
2. Inverse folding reports sequence recovery; generative pipelines report in-silico designability. Both are proxies for 'does it work in the lab'. When is it reasonable to trust a proxy enough to synthesise, and how would you detect that a proxy has stopped tracking reality?
3. Sampling at higher temperature buys novelty at the cost of fitness. For a real design campaign with a fixed number of wells, how would you *allocate* your batch across temperatures (or trust-region radii) rather than picking one?
4. A toxicity model run in reverse can propose harmful molecules (Urbina *et al.* 2022). Where should the safeguards sit — in the model's release, in the optimiser, at the DNA/compound-synthesis step, or in oversight of who may run such loops — and who should be accountable when a safeguard fails?
5. Generative models can memorise and regurgitate training examples. For protein and molecule design, how would you *measure* genuine novelty (versus memorisation), and why does the distinction matter for both scientific credit and safety?

## 🚀 Bonus — going further

- **Bayesian optimisation over sequences** ([Module 29](29_gaussian_processes_bayesian_optimization.ipynb)): replace the greedy surrogate loop in Section 4 with a GP-UCB loop that proposes a batch each round, measures it against the oracle, and refits — a closed DBTL loop. Compare its sample efficiency with simulated annealing on the *true* oracle.
- **Conditional molecule design:** add a hard filter for Lipinski's rule of five (Module 40) inside the GA's objective and see how the Pareto front moves. Then add a crude synthesisability penalty (e.g. penalise many rare substituents) and watch designs become more makeable.
- **Diffusion over a latent space** (preview of [Module 28](28_diffusion_flow_matching.ipynb)): train a small autoencoder on the molecule fingerprints, run a toy diffusion/denoiser in the latent space, and decode samples — the latent-generative-model recipe behind many molecule generators.
- **Conditional sequence design with a grammar** (preview of [Module 36](36_regulatory_genomics_deep_learning.ipynb)): design short regulatory sequences to hit a target activity under the toolkit's `regulatory_activity`, and watch the same out-of-distribution pitfalls appear.
- **Try the real tools** (outside this course's dependencies): run ProteinMPNN on a PDB backbone in ColabDesign, or a REINVENT/GuacaMol baseline on a molecular objective, and compare honest novelty and success against the toy results here.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 39 · Generative design of proteins and molecules
Biological question: can we write new proteins/molecules with desired function, searching huge spaces with imperfect
                     models — safely?
Data (simulated): sample_chemistry_family on simulate_helix_bundle contacts (Potts oracle h, J; wild type); saturating
                  "measured" assay = sigmoid of Potts fitness; simulate_molecules library + noise-free oracle
Method: MCMC sampling at temperature T; simulated annealing + genetic algorithm (oracle); ridge/GP surrogate MBO with
        trust region & pessimism; tiny inverse-folding net (residue from structural neighbours); RF + GA molecule design
        with a Pareto front
Key equation: P(x) ∝ exp(Σ h + Σ J); Metropolis accept exp(ΔE/T); assay y = σ((E−m)/s); LCB = μ − β·σ
Key parameters: temperature T; annealing schedule; GA mutation rate; trust-region radius; GP kernel; multi-objective weight
Ground-truth check result: fitness–diversity trade-off vs T = ___; SA/GA best vs wild type = ___; naive MBO predicted ___
                           vs true ___; trust-region/pessimism true ___; sequence recovery model ___ vs profile ___;
                           designs' oracle fitness ___ vs natural ___; molecule best true pIC50 ___; Pareto front size ___
What changed when I modified … (temperature / mutation rate / trust-region radius / decode temperature / MO weight): …
One failure mode: surrogate drifts out of distribution — predicted soars, true falls
Connection to research: Russ 2020 (evolution-based design); Dauparas 2022 / Watson 2023 (ProteinMPNN / RFdiffusion);
                        Brookes 2019 / Fannjiang 2020 (MBO pitfalls); Gómez-Bombarelli 2018 / Gao 2022 (molecule gen);
                        Urbina 2022 (dual use)
```

## 📚 References

- Huang P.-S., Boyken S. E., Baker D. (2016) The coming of age of de novo protein design. *Nature* 537:320–327.
- Anishchenko I. *et al.* (2021) De novo protein design by deep network hallucination. *Nature* 600:547–552.
- Dauparas J. *et al.* (2022) Robust deep learning-based protein sequence design using ProteinMPNN. *Science* 378:49–56.
- Watson J. L. *et al.* (2023) De novo design of protein structure and function with RFdiffusion. *Nature* 620:1089–1100.
- Ingraham J. B. *et al.* (2023) Illuminating protein space with a programmable generative model (Chroma). *Nature* 623:1070–1078.
- Madani A. *et al.* (2023) Large language models generate functional protein sequences across diverse families (ProGen). *Nat Biotechnol* 41:1099–1106.
- Hayes T. *et al.* (2025) Simulating 500 million years of evolution with a language model (ESM3). *Science* 387:850–858.
- Russ W. P. *et al.* (2020) An evolution-based model for designing chorismate mutase enzymes. *Science* 369:440–445.
- Kirkpatrick S., Gelatt C. D., Vecchi M. P. (1983) Optimization by simulated annealing. *Science* 220:671–680.
- Otwinowski J., McCandlish D. M., Plotkin J. B. (2018) Inferring the shape of global epistasis. *PNAS* 115:E7550–E7558.
- Brookes D. H., Park H., Listgarten J. (2019) Conditioning by adaptive sampling for robust design. *ICML*.
- Fannjiang C., Listgarten J. (2020) Autofocused oracles for model-based design. *NeurIPS*.
- Trabucco B., Kumar A., Geng X., Levine S. (2021) Conservative objective models for effective offline model-based optimization. *ICML*.
- Stanton S. *et al.* (2022) Accelerating Bayesian optimization for biological sequence design with denoising autoencoders. *ICML*.
- Gómez-Bombarelli R. *et al.* (2018) Automatic chemical design using a data-driven continuous representation of molecules. *ACS Cent Sci* 4:268–276.
- Jensen J. H. (2019) A graph-based genetic algorithm and generative model/Monte Carlo tree search for the exploration of chemical space. *Chem Sci* 10:3567–3572.
- Brown N., Fiscato M., Segler M. H. S., Vaucher A. C. (2019) GuacaMol: benchmarking models for de novo molecular design. *J Chem Inf Model* 59:1096–1108.
- Polykovskiy D. *et al.* (2020) Molecular Sets (MOSES): a benchmarking platform for molecular generation models. *Front Pharmacol* 11:565644.
- Gao W., Fu T., Sun J., Coley C. W. (2022) Sample efficiency matters: a benchmark for practical molecular optimization. *NeurIPS Datasets & Benchmarks*.
- Urbina F., Lentzos F., Invernizzi C., Ekins S. (2022) Dual use of artificial-intelligence-powered drug discovery. *Nat Mach Intell* 4:189–191.

## ➡️ Next

We have written proteins and molecules, watched our own models mislead the search, and learned to keep them honest with
trust regions, pessimism and the design–build–test–learn loop. **[Module 40 · Molecular machine learning for drug
discovery](40_molecular_ml_drug_discovery.ipynb)** goes deeper on the molecular surrogates, honest evaluation and
virtual screening that a real design campaign leans on; and the trustworthy-ML and causal-inference modules of Part I
build the calibration, distribution-shift and intervention tools that decide whether a design will survive contact with
reality.

In [ ]:
# ── Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s on {torch.get_num_threads()} CPU thread(s).")